In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, recall_score
from sklearn.metrics import classification_report
from sklearn.metrics import f1_score, make_scorer, log_loss, roc_auc_score
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

In [3]:
df = pd.read_csv("dataset/financial-fraud-detection-dataset/versions/1/Synthetic_Financial_datasets_log.csv")

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 11 columns):
 #   Column          Dtype  
---  ------          -----  
 0   step            int64  
 1   type            object 
 2   amount          float64
 3   nameOrig        object 
 4   oldbalanceOrg   float64
 5   newbalanceOrig  float64
 6   nameDest        object 
 7   oldbalanceDest  float64
 8   newbalanceDest  float64
 9   isFraud         int64  
 10  isFlaggedFraud  int64  
dtypes: float64(5), int64(3), object(3)
memory usage: 534.0+ MB


In [5]:
df = df.drop(columns=['nameOrig', 'nameDest'])
df.head()

,step,type,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,170136.0,160296.36,0.0,0.0,0,0
1,1,PAYMENT,1864.28,21249.0,19384.72,0.0,0.0,0,0
2,1,TRANSFER,181.00,181.0,0.00,0.0,0.0,1,0
3,1,CASH_OUT,181.00,181.0,0.00,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,41554.0,29885.86,0.0,0.0,0,0


In [6]:
X = df.drop(columns=['isFraud', 'isFlaggedFraud'])
y = df['isFraud']

X['is_CASH_IN'] = X['type'].map({'CASH_IN':1, 'PAYMENT':0, 'CASH_OUT':0, 'TRANSFER':0, 'DEBIT':0})
X['is_PAYMENT'] = X['type'].map({'CASH_IN':0, 'PAYMENT':1, 'CASH_OUT':0, 'TRANSFER':0, 'DEBIT':0})
X['is_CASH_OUT'] = X['type'].map({'CASH_IN':0, 'PAYMENT':0, 'CASH_OUT':1, 'TRANSFER':0, 'DEBIT':0})
X['is_TRANSFER'] = X['type'].map({'CASH_IN':0, 'PAYMENT':0, 'CASH_OUT':0, 'TRANSFER':1, 'DEBIT':0})
X['is_DEBIT'] = X['type'].map({'CASH_IN':0, 'PAYMENT':0, 'CASH_OUT':0, 'TRANSFER':0, 'DEBIT':1})

X = X.drop(columns=['type'])

X.head()

,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,is_CASH_IN,is_PAYMENT,is_CASH_OUT,is_TRANSFER,is_DEBIT
0,1,9839.64,170136.0,160296.36,0.0,0.0,0,1,0,0,0
1,1,1864.28,21249.0,19384.72,0.0,0.0,0,1,0,0,0
2,1,181.00,181.0,0.00,0.0,0.0,0,0,0,1,0
3,1,181.00,181.0,0.00,21182.0,0.0,0,0,1,0,0
4,1,11668.14,41554.0,29885.86,0.0,0.0,0,1,0,0,0


In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.7, test_size=0.3, random_state=42)

In [8]:
logreg = LogisticRegression(class_weight='balanced',random_state=42)
logreg.fit(X_train, y_train)


y_pred = logreg.predict(X_test)
y_proba = logreg.predict_proba(X_test)[:,1]

accuracy = accuracy_score(y_test, y_pred)
print(f"Akurasi model Logistic Regression: {accuracy:.2f}")

Akurasi model Logistic Regression: 0.91


In [9]:
logloss_score = log_loss(y_test, y_proba)
print(f"Log loss: {logloss_score:.2f}")
roc_auc = roc_auc_score(y_test, y_proba)
print(f"ROC AUC Score: {roc_auc:.2f}")
y_train_pred = logreg.predict(X_train)
y_test_pred = logreg.predict(X_test)

recall_train = recall_score(y_train, y_train_pred)
recall_test = recall_score(y_test, y_test_pred)

print(f"Training Recall: {recall_train:.2f}")
print(f"Test Recall: {recall_test:.2f}\n")
print(classification_report(y_test, y_pred))

Log loss: 0.33
ROC AUC Score: 0.96
Training Recall: 0.90
Test Recall: 0.88

              precision    recall  f1-score   support

           0       1.00      0.91      0.95   1906351
           1       0.01      0.88      0.02      2435

    accuracy                           0.91   1908786
   macro avg       0.51      0.90      0.49   1908786
weighted avg       1.00      0.91      0.95   1908786



In [10]:
cat_clf = CatBoostClassifier()
rf_clf = RandomForestClassifier()
xgb_clf = XGBClassifier()

voting_clf = VotingClassifier(
    estimators=[('cat', cat_clf), ('rf', rf_clf), ('xgb', xgb_clf)],
    voting='soft'
)

voting_clf.fit(X_train, y_train)

Learning rate set to 0.372343
0:	learn: 0.0621733	total: 713ms	remaining: 11m 52s
1:	learn: 0.0165981	total: 1.27s	remaining: 10m 36s
2:	learn: 0.0102415	total: 1.76s	remaining: 9m 46s
3:	learn: 0.0050278	total: 2.34s	remaining: 9m 42s
4:	learn: 0.0035167	total: 2.87s	remaining: 9m 31s
5:	learn: 0.0026662	total: 3.37s	remaining: 9m 18s
6:	learn: 0.0024240	total: 3.89s	remaining: 9m 12s
7:	learn: 0.0022946	total: 4.45s	remaining: 9m 11s
8:	learn: 0.0022304	total: 4.95s	remaining: 9m 5s
9:	learn: 0.0021243	total: 5.49s	remaining: 9m 3s
10:	learn: 0.0019940	total: 5.99s	remaining: 8m 58s
11:	learn: 0.0019374	total: 6.54s	remaining: 8m 58s
12:	learn: 0.0018850	total: 7.05s	remaining: 8m 55s
13:	learn: 0.0018112	total: 7.55s	remaining: 8m 52s
14:	learn: 0.0017625	total: 8.05s	remaining: 8m 48s
15:	learn: 0.0017337	total: 8.59s	remaining: 8m 48s
16:	learn: 0.0017194	total: 9.1s	remaining: 8m 45s
17:	learn: 0.0016799	total: 9.62s	remaining: 8m 44s
18:	learn: 0.0016130	total: 10.1s	remaining: 

VotingClassifier(estimators=[('cat',
                              <catboost.core.CatBoostClassifier object at 0x0000024D0B564E90>),
                             ('rf', RandomForestClassifier()),
                             ('xgb',
                              XGBClassifier(base_score=None, booster=None,
                                            callbacks=None,
                                            colsample_bylevel=None,
                                            colsample_bynode=None,
                                            colsample_bytree=None, device=None,
                                            early_stopping_rounds=None,
                                            enable_categorical=False,
                                            eval_metric=None,
                                            feature...
                                            importance_type=None,
                                            interaction_constraints=None,
                                            learning_rate=None, max_bin=None,
                                            max_cat_threshold=None,
                                            max_cat_to_onehot=None,
                                            max_delta_step=None, max_depth=None,
                                            max_leaves=None,
                                            min_child_weight=None, missing=nan,
                                            monotone_constraints=None,
                                            multi_strategy=None,
                                            n_estimators=None, n_jobs=None,
                                            num_parallel_tree=None,
                                            random_state=None, ...))],
                 voting='soft')

In [11]:
labels = ['Negatif (0)', 'Positif (1)']

In [12]:
def evaluate_model(classifier, X_train, y_train, X_test, y_test):
    y_train_pred = classifier.predict(X_train)
    y_test_pred = classifier.predict(X_test)
    # on training
    recall_train = recall_score(y_train, y_train_pred)
    # on testing dataset
    recall_test = recall_score(y_test, y_test_pred)

    print(f'Training Recall: {recall_train:.3f}')
    print(f'Test Recall: {recall_test:.3f}')
    print("\nClassification Report (Test):")
    print(classification_report(y_test, y_test_pred))

    cm = confusion_matrix(y_test, y_test_pred)
    
    plt.figure(figsize=(6, 5))
    sns.heatmap(
        cm, 
        annot=True,        
        fmt='d',           
        cmap='Oranges',      
        xticklabels=labels,
        yticklabels=labels
    )
    plt.xlabel('Nilai Prediksi')
    plt.ylabel('Nilai Aktual')
    plt.title(f'Confusion Matrix Heatmap Voting Classifier')
    plt.show()

    return recall_train, recall_test


In [13]:
evaluate_model(voting_clf, X_train, y_train, X_test, y_test)

Training Recall: 0.952
Test Recall: 0.847

Classification Report (Test):
              precision    recall  f1-score   support

           0       1.00      1.00      1.00   1906351
           1       0.98      0.85      0.91      2435

    accuracy                           1.00   1908786
   macro avg       0.99      0.92      0.95   1908786
weighted avg       1.00      1.00      1.00   1908786



NameError: name 'sns' is not defined

<Figure size 600x500 with 0 Axes>

In [ ]:
import joblib

# Save the trained model
joblib.dump(voting_clf, 'voting_classifier_model.pkl')
print("Model saved as 'voting_classifier_model.pkl'")